# Peak Investigation & Cross-Species Comparison

Exploration of the cross-species consensus peak set produced by `04_cross_species_consensus`.

## Contents
1. **Setup & Configuration** — load paths and peak files
2. **Peak Summary** — counts by category (unified, human-specific, species-specific)
3. **Peak Size Analysis** — size distributions per category and across genomes
4. **Species Detection Patterns** — UpSet plots of which species share peaks
5. **Gene Annotation** — closest gene, distance distributions
6. **Gene Conservation** — do lifted peaks land near the same gene in each species?
7. **Liftover Similarity** — round-trip match ratio per peak per species
8. **Cross-Mapping** — species-specific peaks mapped between NHP genomes
9. **Summary & Validation** — ID uniqueness, liftover success rates

## 1. Setup & Configuration

In [ ]:
import sys
import os
import subprocess
import tempfile
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))

from src.cross_species import (
    DEFAULT_GTF_FILES,
    REVERSE_CHAIN_FILES,
    compute_species_similarity,
)
from src.liftover import DEFAULT_CHAIN_DIR
from src.visualization import plot_upset

print("All imports loaded successfully")

In [ ]:
# =============================================================================
# Configuration — must match notebook 04
# =============================================================================

PEAKS_BASE = "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUTPUT_DIR = f"{PEAKS_BASE}/cross_species_consensus_v2"
COMBINED_DIR = os.path.join(OUTPUT_DIR, "09_combined")

LIFTOVER_PATH = "/cluster/project/treutlein/jjans/software/miniforge3/envs/genomes/bin/liftOver"
CHAIN_DIR = DEFAULT_CHAIN_DIR
NCPU = 16

# Species configuration
NHP_SPECIES = ["Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]
ALL_SPECIES = ["Human"] + NHP_SPECIES

ASSEMBLY_MAP = {
    "Human": "hg38",
    "Bonobo": "panPan2",
    "Chimpanzee": "panTro5",
    "Gorilla": "gorGor4",
    "Macaque": "rheMac10",
    "Marmoset": "calJac1",
}

# Original species consensus peaks (for reference)
SPECIES_BEDS = {
    "Bonobo":      f"{PEAKS_BASE}/consensus_peak_calling_Bonobo/Consensus_Peaks_Filtered_500.bed",
    "Chimpanzee":  f"{PEAKS_BASE}/consensus_peak_calling_Chimpanzee/Consensus_Peaks_Filtered_500.bed",
    "Gorilla":     f"{PEAKS_BASE}/consensus_peak_calling_Gorilla/Consensus_Peaks_Filtered_500.bed",
    "Macaque":     f"{PEAKS_BASE}/consensus_peak_calling_Macaque/Consensus_Peaks_Filtered_500.bed",
    "Marmoset":    f"{PEAKS_BASE}/consensus_peak_calling_Marmoset/Consensus_Peaks_Filtered_500.bed",
}

GTF_FILES = DEFAULT_GTF_FILES.copy()

print(f"Output directory: {OUTPUT_DIR}")
print(f"Combined BEDs:   {COMBINED_DIR}")
print(f"Species:         {ALL_SPECIES}")

## 2. Load Data

Load the combined BED files (`all_peaks_{Species}.bed`) and master annotation from notebook 04.

In [ ]:
# =============================================================================
# Load combined BED files (all_peaks_{Species}.bed)
# These contain unified + species-specific peaks per species
# =============================================================================

combined_beds = {}  # species -> DataFrame(chr, start, end, peak_id)

print("Loading combined BED files (unified + species-specific per genome):")
print("=" * 70)

for species in ALL_SPECIES:
    bed_path = os.path.join(COMBINED_DIR, f"all_peaks_{species}.bed")
    if os.path.exists(bed_path):
        df = pd.read_csv(bed_path, sep="\t", header=None,
                         names=["chr", "start", "end", "peak_id"])
        df["size"] = df["end"] - df["start"]
        # Classify peak type from ID prefix
        df["peak_type"] = df["peak_id"].apply(
            lambda x: "unified" if x.startswith("unified_")
            else "human_specific" if x.startswith("human_peak_")
            else "species_specific"
        )
        combined_beds[species] = df
        n_uni = (df["peak_type"] == "unified").sum()
        n_sp = (df["peak_type"] != "unified").sum()
        print(f"  {species:<15s} {len(df):>8,} peaks  "
              f"(unified: {n_uni:,}, specific: {n_sp:,})  "
              f"[{ASSEMBLY_MAP[species]}]")
    else:
        print(f"  {species:<15s} NOT FOUND: {bed_path}")

print(f"\nLoaded {len(combined_beds)}/{len(ALL_SPECIES)} species")

In [ ]:
# =============================================================================
# Load master annotation (comprehensive per-peak metadata)
# =============================================================================

master_file = os.path.join(OUTPUT_DIR, "07_master_annotation", "master_annotation.tsv")
master = pd.read_csv(master_file, sep="\t", index_col="peak_id")

print(f"Master annotation: {len(master):,} peaks x {len(master.columns)} columns")
print(f"\nPeak types:")
print(master["peak_type"].value_counts())

# Detection & orthology columns
det_cols = [c for c in master.columns if c.endswith("_det")]
orth_cols = [c for c in master.columns if c.endswith("_orth")]

print(f"\nDetection columns: {det_cols}")
print(f"Orthology columns: {orth_cols}")

# Show sample
show_cols = ["peak_type", "n_species_det", "n_species_orth"] + det_cols + orth_cols
display(master[show_cols].head(10))

In [ ]:
# =============================================================================
# Load unified-only BED files for cross-genome peak size comparison
# =============================================================================

unified_beds = {}  # species -> DataFrame of unified-only peaks

# Human: unified peaks in hg38
uni_hg38_path = os.path.join(COMBINED_DIR, "unified_peaks_hg38.bed")
if os.path.exists(uni_hg38_path):
    unified_beds["Human"] = pd.read_csv(
        uni_hg38_path, sep="\t", header=None,
        names=["chr", "start", "end", "peak_id"]
    )
    unified_beds["Human"]["size"] = unified_beds["Human"]["end"] - unified_beds["Human"]["start"]

# NHP: unified peaks lifted to species coords
for species in NHP_SPECIES:
    uni_sp_path = os.path.join(COMBINED_DIR, f"unified_peaks_{species}.bed")
    if os.path.exists(uni_sp_path):
        unified_beds[species] = pd.read_csv(
            uni_sp_path, sep="\t", header=None,
            names=["chr", "start", "end", "peak_id"]
        )
        unified_beds[species]["size"] = unified_beds[species]["end"] - unified_beds[species]["start"]

print("Unified-only BED files (for cross-genome size comparison):")
for sp, df in unified_beds.items():
    print(f"  {sp:<15s} {len(df):>8,} unified peaks  "
          f"(median size: {df['size'].median():.0f} bp)")

## 3. Peak Summary

Overview of peak counts by category across all species/genomes.

In [ ]:
# =============================================================================
# Peak count summary table
# =============================================================================

summary_rows = []
for species in ALL_SPECIES:
    if species not in combined_beds:
        continue
    df = combined_beds[species]
    n_unified = (df["peak_type"] == "unified").sum()
    n_hs = (df["peak_type"] == "human_specific").sum()
    n_sp = (df["peak_type"] == "species_specific").sum()
    summary_rows.append({
        "Species": species,
        "Genome": ASSEMBLY_MAP[species],
        "Unified": n_unified,
        "Human-specific": n_hs,
        "Species-specific": n_sp,
        "Total": len(df),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Species")
print("Peak counts per species/genome:")
print("=" * 80)
display(summary_df)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary_df))
w = 0.25
ax.bar(x - w, summary_df["Unified"], w, label="Unified", color="steelblue")
ax.bar(x,     summary_df["Human-specific"] + summary_df["Species-specific"],
       w, label="Specific", color="coral")
ax.bar(x + w, summary_df["Total"], w, label="Total", color="gray", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(summary_df.index, rotation=30, ha="right")
ax.set_ylabel("Number of peaks")
ax.set_title("Peak Counts per Species/Genome")
ax.legend()
for i, total in enumerate(summary_df["Total"]):
    ax.text(i + w, total, f"{total:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## 4. Peak Size Analysis

Peak sizes vary because:
- **Unified peaks in hg38**: fixed-width (summit_window, default 500 bp)
- **Unified peaks after liftback**: sizes change due to indels/rearrangements during liftover
- **Species-specific peaks**: original consensus peak sizes (variable)
- **Human-specific peaks**: from the original human consensus (variable)

In [ ]:
# =============================================================================
# Size distributions per category
# =============================================================================

print("Peak size summary by category (all species combined):")
print("=" * 80)

all_peaks = pd.concat(
    [df.assign(species=sp) for sp, df in combined_beds.items()],
    ignore_index=True,
)

# Summary stats per category
size_stats = all_peaks.groupby("peak_type")["size"].describe()
display(size_stats)

# Histograms per category
categories = all_peaks["peak_type"].unique()
fig, axes = plt.subplots(1, len(categories), figsize=(6 * len(categories), 4),
                         sharey=True)
if len(categories) == 1:
    axes = [axes]

colors = {"unified": "steelblue", "human_specific": "firebrick",
          "species_specific": "seagreen"}

for ax, cat in zip(axes, sorted(categories)):
    vals = all_peaks.loc[all_peaks["peak_type"] == cat, "size"]
    ax.hist(vals.clip(upper=2000), bins=100, color=colors.get(cat, "gray"),
            alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.axvline(vals.median(), color="black", linestyle="--",
               label=f"median={vals.median():.0f}")
    ax.set_title(f"{cat}\n(n={len(vals):,})")
    ax.set_xlabel("Peak size (bp)")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Count")
plt.suptitle("Peak Size Distributions by Category", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Size of unified peaks per genome (cross-genome comparison)
# =============================================================================

print("Unified peak sizes per genome:")
print("=" * 80)
print(f"{'Species':<15s} {'Genome':<10s} {'N peaks':>10s} {'Median':>10s} "
      f"{'Mean':>10s} {'Min':>8s} {'Max':>8s} {'Std':>10s}")
print("-" * 90)

for sp in ALL_SPECIES:
    if sp in unified_beds:
        sizes = unified_beds[sp]["size"]
        print(f"{sp:<15s} {ASSEMBLY_MAP[sp]:<10s} {len(sizes):>10,} "
              f"{sizes.median():>10.0f} {sizes.mean():>10.1f} "
              f"{sizes.min():>8,} {sizes.max():>8,} {sizes.std():>10.1f}")

# Violin / box plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot of unified peak sizes per genome
species_order = [sp for sp in ALL_SPECIES if sp in unified_beds]
box_data = [unified_beds[sp]["size"].values for sp in species_order]

bp = axes[0].boxplot(box_data, labels=species_order, showfliers=False, patch_artist=True)
genome_colors = {"Human": "#377eb8", "Chimpanzee": "#4daf4a", "Bonobo": "#ff7f00",
                 "Gorilla": "#984ea3", "Macaque": "#e41a1c", "Marmoset": "#a65628"}
for patch, sp in zip(bp["boxes"], species_order):
    patch.set_facecolor(genome_colors.get(sp, "gray"))
    patch.set_alpha(0.7)
axes[0].set_ylabel("Peak size (bp)")
axes[0].set_title("Unified Peak Sizes per Genome\n(after liftover)")
axes[0].tick_params(axis="x", rotation=30)

# Overlaid histograms
for sp in species_order:
    sizes = unified_beds[sp]["size"]
    axes[1].hist(sizes.clip(upper=2000), bins=80, alpha=0.4,
                 label=f"{sp} (med={sizes.median():.0f})",
                 color=genome_colors.get(sp, "gray"), density=True)
axes[1].set_xlabel("Peak size (bp)")
axes[1].set_ylabel("Density")
axes[1].set_title("Unified Peak Size Distributions (Overlaid)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Per-peak size comparison across genomes (using master annotation coords)
# =============================================================================

# For unified peaks, the master annotation has {Species}_start/{Species}_end
# We can compute the size in each genome for the SAME peak

unified_master = master[master["peak_type"] == "unified"].copy()

size_cols = {}
for sp in ALL_SPECIES:
    start_col = f"{sp}_start"
    end_col = f"{sp}_end"
    if start_col in unified_master.columns and end_col in unified_master.columns:
        col_name = f"{sp}_size"
        unified_master[col_name] = unified_master[end_col] - unified_master[start_col]
        size_cols[sp] = col_name

print("Per-peak size in each genome (unified peaks only):")
print("=" * 80)

# Summary table
for sp, col in size_cols.items():
    valid = unified_master[col].dropna()
    print(f"  {sp:<15s} ({ASSEMBLY_MAP[sp]:<10s}): "
          f"n={len(valid):>8,}  median={valid.median():>8.0f}  "
          f"mean={valid.mean():>8.1f}  std={valid.std():>8.1f}")

# Pairwise size scatter: human vs each species
fig, axes = plt.subplots(1, len(NHP_SPECIES), figsize=(4 * len(NHP_SPECIES), 4),
                         sharey=True, sharex=True)

human_size_col = size_cols.get("Human")
if human_size_col:
    for i, sp in enumerate(NHP_SPECIES):
        ax = axes[i]
        sp_col = size_cols.get(sp)
        if sp_col is None:
            ax.set_title(f"{sp}\n(no data)")
            continue

        valid = unified_master[[human_size_col, sp_col]].dropna()
        if len(valid) == 0:
            ax.set_title(f"{sp}\n(no overlapping peaks)")
            continue

        sample = valid.sample(min(5000, len(valid)), random_state=42)
        ax.scatter(sample[human_size_col], sample[sp_col],
                   alpha=0.1, s=2, color=genome_colors.get(sp, "gray"),
                   rasterized=True)
        ax.plot([0, 2000], [0, 2000], "k--", alpha=0.5, lw=0.8)
        ax.set_title(f"{sp} ({ASSEMBLY_MAP[sp]})\n"
                     f"r={valid[human_size_col].corr(valid[sp_col]):.3f}, "
                     f"n={len(valid):,}")
        ax.set_xlabel("Human size (bp)")
        if i == 0:
            ax.set_ylabel("Species size (bp)")
        ax.set_xlim(0, 2000)
        ax.set_ylim(0, 2000)

plt.suptitle("Per-Peak Size: Human vs Species (Unified Peaks)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Size ratio: species / human  (how much does liftover distort peak size?)
# =============================================================================

if human_size_col:
    fig, axes = plt.subplots(1, len(NHP_SPECIES),
                             figsize=(4 * len(NHP_SPECIES), 4), sharey=True)

    for i, sp in enumerate(NHP_SPECIES):
        ax = axes[i]
        sp_col = size_cols.get(sp)
        if sp_col is None:
            continue

        valid = unified_master[[human_size_col, sp_col]].dropna()
        ratio = valid[sp_col] / valid[human_size_col]
        ratio = ratio.replace([np.inf, -np.inf], np.nan).dropna()

        ax.hist(ratio.clip(0, 3), bins=100, color=genome_colors.get(sp, "gray"),
                alpha=0.8, edgecolor="white", linewidth=0.3)
        ax.axvline(1.0, color="red", linestyle="--", alpha=0.7)
        ax.axvline(ratio.median(), color="black", linestyle="-", alpha=0.7,
                   label=f"median={ratio.median():.3f}")
        pct_similar = ((ratio >= 0.8) & (ratio <= 1.2)).mean() * 100
        ax.set_title(f"{sp}\n{pct_similar:.0f}% within 0.8–1.2x")
        ax.set_xlabel("Size ratio (species / human)")
        ax.legend(fontsize=8)

    axes[0].set_ylabel("Count")
    plt.suptitle("Liftover Size Distortion (Species / Human)", fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# Species-specific peak sizes per species
# =============================================================================

print("Species-specific peak sizes:")
print("=" * 70)

sp_specific_sizes = {}
for species in ALL_SPECIES:
    if species not in combined_beds:
        continue
    df = combined_beds[species]
    specific = df[df["peak_type"] != "unified"]
    if len(specific) > 0:
        sp_specific_sizes[species] = specific["size"]
        print(f"  {species:<15s}: {len(specific):>8,} peaks, "
              f"median={specific['size'].median():.0f} bp, "
              f"mean={specific['size'].mean():.1f} bp, "
              f"range=[{specific['size'].min()}, {specific['size'].max()}]")

if sp_specific_sizes:
    fig, ax = plt.subplots(figsize=(10, 5))
    for sp, sizes in sp_specific_sizes.items():
        ax.hist(sizes.clip(upper=2000), bins=80, alpha=0.5, density=True,
                label=f"{sp} (n={len(sizes):,}, med={sizes.median():.0f})",
                color=genome_colors.get(sp, "gray"))
    ax.set_xlabel("Peak size (bp)")
    ax.set_ylabel("Density")
    ax.set_title("Species-Specific Peak Size Distributions")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Species Detection Patterns

UpSet plot showing which species combinations share peaks.
Each intersection represents a unique combination of species in which a unified peak was detected.

In [ ]:
# =============================================================================
# Load unified consensus for detection analysis
# =============================================================================

unified_bed_path = os.path.join(OUTPUT_DIR, "02_merged_consensus",
                                "unified_consensus_hg38_with_ids.bed")
unified_df = pd.read_csv(unified_bed_path, sep="\t", header=None,
                         names=["chr", "start", "end", "peak_id", "species_detected"])

print(f"Unified consensus peaks: {len(unified_df):,}")
print(f"\nFirst 10 peaks:")
print(unified_df.head(10).to_string(index=False))

# Parse species detection
for sp in ALL_SPECIES:
    unified_df[f"detected_in_{sp}"] = unified_df["species_detected"].str.contains(sp, na=False)

unified_df["n_species_det"] = unified_df[
    [f"detected_in_{sp}" for sp in ALL_SPECIES]
].sum(axis=1)

print(f"\nSpecies detection distribution:")
print(unified_df["n_species_det"].value_counts().sort_index().to_string())

print(f"\nPer-species detection:")
for sp in ALL_SPECIES:
    n = unified_df[f"detected_in_{sp}"].sum()
    pct = n / len(unified_df) * 100
    print(f"  {sp:<15s}: {n:>10,} peaks ({pct:5.1f}%)")

In [ ]:
# =============================================================================
# UpSet plot: species detection overlap
# =============================================================================

species_cols = [f"detected_in_{sp}" for sp in ALL_SPECIES]

plot_file = os.path.join(OUTPUT_DIR, "upset_species_detection.png")

fig = plot_upset(
    unified_df,
    set_columns=species_cols,
    set_labels=ALL_SPECIES,
    top_n=30,
    color="steelblue",
    title="Peak Detection Across Primate Species",
    saveas=plot_file,
)

# Print top intersections
upset_df = unified_df[species_cols].copy()
upset_df.columns = ALL_SPECIES
pattern = upset_df.astype(int).apply(tuple, axis=1)
combo_counts = pattern.value_counts().sort_values(ascending=False)

print(f"\nTop 15 intersections:")
for combo_tuple, count in combo_counts.head(15).items():
    names = [sp for sp, flag in zip(ALL_SPECIES, combo_tuple) if flag]
    pct = count / len(unified_df) * 100
    print(f"  {', '.join(names):<55s} {count:>8,} peaks ({pct:5.1f}%)")

In [ ]:
# =============================================================================
# UpSet plot from master annotation (all peak types)
# =============================================================================

det_cols = [c for c in master.columns if c.endswith("_det")]
species_names = [c.replace("_det", "") for c in det_cols]

fig = plot_upset(
    master,
    set_columns=det_cols,
    set_labels=species_names,
    top_n=30,
    color="steelblue",
    title="Species Detection Patterns \u2014 All Peaks (Master Annotation)",
)

# Gene distance comparison across species
gene_dist_cols = [c for c in master.columns if c.endswith("_gene_dist")]
if gene_dist_cols:
    fig, ax = plt.subplots(figsize=(10, 5))
    for col in gene_dist_cols:
        sp = col.replace("_gene_dist", "")
        vals = master[col].dropna()
        if len(vals) > 0:
            vals_kb = vals[vals >= 0] / 1000
            ax.hist(vals_kb.clip(upper=500), bins=100, alpha=0.5,
                    label=f"{sp} (n={len(vals):,})")
    ax.set_xlabel("Distance to nearest gene (kb)")
    ax.set_ylabel("Count")
    ax.set_title("Distance to Nearest Gene by Species")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Explore Human-Specific and Species-Specific Peaks

In [ ]:
# =============================================================================
# Human-specific peaks
# =============================================================================

hs_bed = os.path.join(COMBINED_DIR, "human_specific_peaks_hg38.bed")
if os.path.exists(hs_bed):
    hs_df = pd.read_csv(hs_bed, sep="\t", header=None,
                         names=["chr", "start", "end", "peak_id"])
    print(f"Human-specific peaks: {len(hs_df):,}")
    print(f"First 5:")
    print(hs_df.head().to_string(index=False))
else:
    hs_df = pd.DataFrame()
    print("Human-specific BED not found")

# =============================================================================
# Species-specific peaks
# =============================================================================
print(f"\n{'='*70}")
print(f"Species-specific peaks:")
print(f"{'='*70}")

species_specific_dir = os.path.join(OUTPUT_DIR, "05_species_specific")
species_specific_counts = {}

for species in NHP_SPECIES:
    sp_bed = os.path.join(species_specific_dir, f"{species}_specific_peaks.bed")
    if os.path.exists(sp_bed):
        sp_df = pd.read_csv(sp_bed, sep="\t", header=None,
                            names=["chr", "start", "end", "peak_id"])
        species_specific_counts[species] = len(sp_df)
        print(f"\n  {species}: {len(sp_df):,} specific peaks")
        print(f"  Example IDs: {', '.join(sp_df['peak_id'].head(3))}")

# Summary
print(f"\n{'='*70}")
print(f"Summary:")
print(f"  Unified consensus:   {len(unified_df):>10,}")
print(f"  Human-specific:      {len(hs_df):>10,}")
for sp, n in species_specific_counts.items():
    print(f"  {sp}-specific: {n:>10,}")
total = len(unified_df) + len(hs_df) + sum(species_specific_counts.values())
print(f"  {'TOTAL':>22s}: {total:>10,}")

## 7. Gene Annotation Overview

In [ ]:
# =============================================================================
# Gene annotation overview — from master annotation (orthology-based)
# =============================================================================
# NOTE: We use the master annotation (07_master_annotation/master_annotation.tsv)
# rather than the intermediate 06_annotation/peak_annotation.tsv, because only
# the master annotation has the correct orthology-based peak classification
# (unified vs human_specific).

# master_df was loaded earlier; use it directly
annot_df = master_df.copy()
if annot_df.index.name == "peak_id":
    annot_df = annot_df.reset_index()

print(f"Total annotated peaks: {len(annot_df):,}")
print(f"\nPeak types (orthology-based classification):")
print(annot_df["peak_type"].value_counts().to_string())

# Gene distance summary for unified + human_specific peaks
gene_col = "Human_gene_dist" if "Human_gene_dist" in annot_df.columns else None
if gene_col:
    valid_dist = annot_df.loc[annot_df[gene_col] >= 0, gene_col]
    print(f"\nDistance to nearest gene — Human annotation (summary):")
    print(f"  Median: {valid_dist.median():,.0f} bp")
    print(f"  Mean:   {valid_dist.mean():,.0f} bp")
    print(f"  At TSS (0 bp): {(valid_dist == 0).sum():,}")
    print(f"  < 1 kb:  {(valid_dist < 1000).sum():,}")
    print(f"  < 10 kb: {(valid_dist < 10000).sum():,}")
    print(f"  < 100 kb: {(valid_dist < 100000).sum():,}")

print(f"\nSample rows from each peak type:")
display_cols = [c for c in ["peak_id", "peak_type", "Human_chr", "Human_start",
                             "Human_end", "species_detected", "n_species_det",
                             "n_species_orth", "Human_gene"] if c in annot_df.columns]
for pt in annot_df["peak_type"].unique():
    subset = annot_df[annot_df["peak_type"] == pt].head(3)
    print(f"\n  {pt}:")
    print(subset[display_cols].to_string(index=False))

## 8. Gene Conservation After Liftback

For each unified peak we know the closest human gene (hg38 annotation).
After lifting those same peaks back to each species\u2019 genome, we can ask:
**is the closest gene in the species genome the same gene?**

This checks whether the syntenic neighbourhood is preserved across the liftover round-trip.
Gene name matching is case-insensitive and uses the gene symbol (not Ensembl ID),
so conservation rates are a lower bound.

In [ ]:
# =============================================================================
# Gene conservation analysis: human annotation vs species liftback annotation
# =============================================================================

# Load human gene annotation for unified peaks
human_annot = pd.read_csv(
    os.path.join(OUTPUT_DIR, "06_annotation", "unified_gene_annotation.tsv"),
    sep="\t",
)
print(f"Human annotation: {len(human_annot):,} unified peaks with closest gene")
n_human_ens = human_annot["closest_gene"].str.startswith("ENS").sum()
n_human_named = len(human_annot) - n_human_ens
print(f"  Human gene symbols: {n_human_named:,} ({n_human_named/len(human_annot)*100:.0f}%)")
print(f"  Human Ensembl IDs:  {n_human_ens:,} ({n_human_ens/len(human_annot)*100:.0f}%)")

# Gene BED directory (already extracted by pipeline)
gene_bed_dir = os.path.join(OUTPUT_DIR, "06_annotation", "gene_beds")
liftback_dir = os.path.join(OUTPUT_DIR, "04_lifted_back")

conservation_results = {}

for species in NHP_SPECIES:
    liftback_file = os.path.join(liftback_dir, f"unified_consensus_{species}.bed")
    gene_bed = os.path.join(gene_bed_dir, f"{species}_genes.bed")

    if not os.path.exists(liftback_file) or not os.path.exists(gene_bed):
        print(f"  Skipping {species} - missing files")
        continue

    sp_n_genes = sum(1 for _ in open(gene_bed))

    with tempfile.TemporaryDirectory() as tmpdir:
        with open(liftback_file) as f:
            lb_chrom = f.readline().split('\t')[0]
        with open(gene_bed) as f:
            gene_chrom = f.readline().split('\t')[0]

        lb_has_chr = lb_chrom.startswith("chr")
        gene_has_chr = gene_chrom.startswith("chr")

        gene_bed_use = gene_bed
        if lb_has_chr and not gene_has_chr:
            gene_bed_use = os.path.join(tmpdir, "genes_chr.bed")
            with open(gene_bed) as fin, open(gene_bed_use, 'w') as fout:
                for line in fin:
                    if line.strip():
                        fout.write("chr" + line)
        elif not lb_has_chr and gene_has_chr:
            gene_bed_use = os.path.join(tmpdir, "genes_nochr.bed")
            with open(gene_bed) as fin, open(gene_bed_use, 'w') as fout:
                for line in fin:
                    if line.strip():
                        fout.write(line[3:] if line.startswith("chr") else line)

        lb_sorted = os.path.join(tmpdir, "lb_sorted.bed")
        gene_sorted = os.path.join(tmpdir, "gene_sorted.bed")
        subprocess.run(f"sort -k1,1 -k2,2n {liftback_file} > {lb_sorted}",
                       shell=True, check=True)
        subprocess.run(f"sort -k1,1 -k2,2n {gene_bed_use} > {gene_sorted}",
                       shell=True, check=True)

        closest_out = os.path.join(tmpdir, "closest.tsv")
        subprocess.run(
            f"bedtools closest -a {lb_sorted} -b {gene_sorted} -d -t first > {closest_out}",
            shell=True, check=True,
        )

        with open(lb_sorted) as f:
            n_lb_cols = len(f.readline().strip().split('\t'))

        rows = []
        with open(closest_out) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < n_lb_cols + 7:
                    continue
                peak_id = parts[3]
                sp_gene = parts[n_lb_cols + 3]
                sp_dist = int(parts[-1]) if parts[-1] != '.' else -1
                rows.append({"peak_id": peak_id, "sp_gene": sp_gene,
                             "sp_distance": sp_dist})

    sp_annot = pd.DataFrame(rows)
    merged = human_annot.merge(sp_annot, on="peak_id", how="inner")
    merged = merged[merged["sp_distance"] >= 0]

    merged["human_is_ens"] = merged["closest_gene"].str.startswith("ENS")
    merged["sp_is_ens"] = merged["sp_gene"].str.startswith("ENS")
    merged["same_gene"] = merged["closest_gene"].str.upper() == merged["sp_gene"].str.upper()

    both_named = ~merged["human_is_ens"] & ~merged["sp_is_ens"]
    both_ens = merged["human_is_ens"] & merged["sp_is_ens"]
    mixed = (merged["human_is_ens"] & ~merged["sp_is_ens"]) | (
        ~merged["human_is_ens"] & merged["sp_is_ens"]
    )

    near_gene = (merged["distance_to_gene"] < 10000) & (merged["sp_distance"] < 10000)
    near_both_named = both_named & near_gene

    n_total = len(merged)
    n_both_named = both_named.sum()
    n_same_named = merged.loc[both_named, "same_gene"].sum()
    n_near_both_named = near_both_named.sum()
    n_same_near = merged.loc[near_both_named, "same_gene"].sum()

    conservation_results[species] = {
        "total_peaks": n_total,
        "sp_n_genes": sp_n_genes,
        "both_named": n_both_named,
        "same_named": n_same_named,
        "pct_named": n_same_named / n_both_named * 100 if n_both_named else 0,
        "near_both_named": n_near_both_named,
        "same_near": n_same_near,
        "pct_near": n_same_near / n_near_both_named * 100 if n_near_both_named else 0,
        "both_ens": both_ens.sum(),
        "mixed": mixed.sum(),
    }

    print(f"\n{species} ({sp_n_genes:,} gene TSSs in GTF):")
    print(f"  Peaks compared:                      {n_total:>10,}")
    if n_both_named:
        print(f"  Both have gene symbol:               {n_both_named:>10,}  "
              f"-> same gene: {n_same_named:,} ({n_same_named/n_both_named*100:.1f}%)")
    if n_near_both_named:
        print(f"  Both named + both <10kb from gene:   {n_near_both_named:>10,}  "
              f"-> same gene: {n_same_near:,} ({n_same_near/n_near_both_named*100:.1f}%)")
    print(f"  Both Ensembl IDs (can't compare):    {both_ens.sum():>10,}")
    print(f"  Mixed (Ensembl vs symbol):            {mixed.sum():>10,}")

# Summary table
print(f"\n{'='*80}")
print(f"GENE CONSERVATION SUMMARY")
print(f"{'='*80}")
print(f"  {'Species':<15s} {'Genes':>7s} {'Both named':>12s} {'Same':>8s} "
      f"{'%':>6s}  {'Near(<10kb)':>12s} {'Same':>8s} {'%':>6s}")
print(f"  {'-'*78}")
for species, r in conservation_results.items():
    print(f"  {species:<15s} {r['sp_n_genes']:>7,} {r['both_named']:>12,} "
          f"{r['same_named']:>8,} {r['pct_named']:>5.1f}%  "
          f"{r['near_both_named']:>12,} {r['same_near']:>8,} {r['pct_near']:>5.1f}%")

In [ ]:
# =============================================================================
# Visualization: gene conservation
# =============================================================================

if conservation_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 1. Bar chart: conservation rate by species
    species_names = list(conservation_results.keys())
    pct_named = [conservation_results[s]["pct_named"] for s in species_names]
    pct_near = [conservation_results[s]["pct_near"] for s in species_names]

    x = np.arange(len(species_names))
    w = 0.35
    axes[0].bar(x - w/2, pct_named, w, label="Both named (all dist.)",
                color="steelblue")
    axes[0].bar(x + w/2, pct_near, w, label="Both named + both <10kb",
                color="coral")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(species_names, rotation=30, ha="right")
    axes[0].set_ylabel("% same closest gene")
    axes[0].set_title("Gene conservation after liftback")
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0, 100)

    # 2. Summary bar: n peaks per species
    n_peaks_arr = [conservation_results[s]["total_peaks"] for s in species_names]
    axes[1].barh(species_names, n_peaks_arr, color="steelblue", alpha=0.7)
    axes[1].set_xlabel("Number of peaks compared")
    axes[1].set_title("Liftback peaks per species")
    for i, n in enumerate(n_peaks_arr):
        axes[1].text(n, i, f"  {n:,}", ha="left", va="center", fontsize=8)

    plt.tight_layout()
    plt.show()

## 9. Per-Peak Liftover Similarity

For each unified hg38 peak, a **round-trip liftover** (hg38 \u2192 species \u2192 hg38) measures the
fraction of original bases recovered. This gives a continuous **match_ratio** (0\u20131) per peak per species.

In [ ]:
# =============================================================================
# Load or compute liftover similarity
# =============================================================================

similarity_dir = os.path.join(OUTPUT_DIR, "10_similarity")
matrix_tsv = os.path.join(similarity_dir, "match_ratio_matrix.tsv")

if os.path.exists(matrix_tsv):
    match_matrix = pd.read_csv(matrix_tsv, sep="\t", index_col="peak_id")
    print(f"Loaded match ratio matrix: {match_matrix.shape}")
else:
    print(f"Match ratio matrix not found at {matrix_tsv}")
    print("Computing from scratch...")

    import importlib
    import atac_pipeline.src.liftover as lo_mod
    import atac_pipeline.src.cross_species as cs_mod
    importlib.reload(lo_mod)
    importlib.reload(cs_mod)
    from atac_pipeline.src.cross_species import compute_species_similarity

    os.makedirs(similarity_dir, exist_ok=True)
    output_tsv = os.path.join(similarity_dir, "liftover_similarity_all_species.tsv")

    similarity_df = compute_species_similarity(
        input_bed=unified_bed_path,
        species_list=NHP_SPECIES,
        liftover_path=LIFTOVER_PATH,
        output_tsv=output_tsv,
        verbose=True,
        ncpu=NCPU,
    )

    # Pivot to matrix
    match_matrix = similarity_df.pivot(
        index="peak_id", columns="species", values="match_ratio"
    )
    match_matrix["Human"] = 1.0
    col_order = ["Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset"]
    match_matrix = match_matrix[[c for c in col_order if c in match_matrix.columns]]

    coords = similarity_df[similarity_df["species"] == similarity_df["species"].iloc[0]][
        ["peak_id", "chrom", "start", "end", "original_size"]
    ].set_index("peak_id")
    match_matrix = coords.join(match_matrix)
    match_matrix.to_csv(matrix_tsv, sep="\t")
    print(f"Saved: {matrix_tsv}")

# Summary stats
species_ratio_cols = [c for c in match_matrix.columns if c in NHP_SPECIES]
print(f"\nPer-species summary (median match_ratio):")
for sp in species_ratio_cols:
    vals = match_matrix[sp].dropna()
    mapped = vals[vals > 0]
    print(f"   {sp:15s}: median={vals.median():.3f}  mean={vals.mean():.3f}  "
          f"mapped={len(mapped):,}/{len(vals):,} ({len(mapped)/len(vals)*100:.1f}%)  "
          f"\u22650.95: {(vals >= 0.95).sum():,}")

display(match_matrix.head(10))

In [ ]:
# =============================================================================
# Match ratio distributions per species
# =============================================================================

species_ratio_cols = [c for c in ["Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset"]
                      if c in match_matrix.columns]

fig, axes = plt.subplots(1, len(species_ratio_cols),
                         figsize=(4 * len(species_ratio_cols), 4), sharey=True)

for i, sp in enumerate(species_ratio_cols):
    ax = axes[i]
    vals = match_matrix[sp].dropna()

    ax.hist(vals, bins=100, range=(0, 1.05),
            color=genome_colors.get(sp, "gray"), alpha=0.8,
            edgecolor="white", linewidth=0.3)
    ax.axvline(0.95, color="red", linestyle="--", alpha=0.7, label="0.95")
    ax.axvline(vals.median(), color="black", linestyle="-", alpha=0.7,
               label=f"med={vals.median():.3f}")

    pct95 = (vals >= 0.95).mean() * 100
    ax.set_title(f"{sp}\n\u22650.95: {pct95:.1f}%", fontsize=11)
    ax.set_xlabel("Match ratio")
    if i == 0:
        ax.set_ylabel("# peaks")
    ax.legend(fontsize=7)
    ax.set_xlim(-0.02, 1.05)

plt.suptitle("Per-Peak Match Ratio to hg38 (Round-Trip)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(similarity_dir, "match_ratio_distributions.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 10. Cross-Mapping Species-Specific Peaks

Species-specific peaks failed the hg38 round-trip. We use **direct inter-species chain files**
to check whether they overlap open-chromatin regions in other species.

In [ ]:
# =============================================================================
# Cross-mapping results
# =============================================================================

cross_map_file = os.path.join(OUTPUT_DIR, "08_cross_mapping",
                              "species_specific_cross_mapping.tsv")
cross_matrix_file = os.path.join(OUTPUT_DIR, "08_cross_mapping",
                                 "cross_mapping_matrix_pct.tsv")
cross_counts_file = os.path.join(OUTPUT_DIR, "08_cross_mapping",
                                 "cross_mapping_matrix_counts.tsv")

if os.path.exists(cross_map_file):
    cross_map = pd.read_csv(cross_map_file, sep="\t")
    print(f"Cross-mapping results: {len(cross_map)} source-target pairs\n")

    show_cols = ["source", "target", "source_specific", "n_hops",
                 "lifted_to_target", "overlap_target_peaks", "pct_overlap"]
    display(cross_map[show_cols])

    if os.path.exists(cross_matrix_file):
        matrix = pd.read_csv(cross_matrix_file, sep="\t", index_col=0)
        print(f"\nCross-mapping matrix (% overlap):")
        display(matrix.round(1))

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Percentage heatmap
        ax = axes[0]
        im = ax.imshow(matrix.values, cmap="YlOrRd", aspect="auto")
        ax.set_xticks(range(len(matrix.columns)))
        ax.set_xticklabels(matrix.columns, rotation=45, ha="right")
        ax.set_yticks(range(len(matrix.index)))
        ax.set_yticklabels(matrix.index)
        ax.set_xlabel("Target species")
        ax.set_ylabel("Source species")
        ax.set_title("% of Species-Specific Peaks\nOverlapping Target Peaks")
        for i in range(len(matrix.index)):
            for j in range(len(matrix.columns)):
                val = matrix.values[i, j]
                color = "white" if val > 50 else "black"
                ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                        color=color, fontsize=9)
        plt.colorbar(im, ax=ax, label="% overlap")

        # Count heatmap
        if os.path.exists(cross_counts_file):
            counts = pd.read_csv(cross_counts_file, sep="\t", index_col=0)
            ax2 = axes[1]
            im2 = ax2.imshow(counts.values, cmap="Blues", aspect="auto")
            ax2.set_xticks(range(len(counts.columns)))
            ax2.set_xticklabels(counts.columns, rotation=45, ha="right")
            ax2.set_yticks(range(len(counts.index)))
            ax2.set_yticklabels(counts.index)
            ax2.set_xlabel("Target species")
            ax2.set_ylabel("Source species")
            ax2.set_title("Absolute Count of Overlapping Peaks")
            for i in range(len(counts.index)):
                for j in range(len(counts.columns)):
                    val = int(counts.values[i, j])
                    color = "white" if val > counts.values.max() * 0.6 else "black"
                    ax2.text(j, i, f"{val:,}", ha="center", va="center",
                             color=color, fontsize=9)
            plt.colorbar(im2, ax=ax2, label="count")

        plt.tight_layout()
        plt.show()
else:
    print(f"Cross-mapping not found at {cross_map_file}")
    print("Run notebook 04 first.")

## 11. Summary & Validation

In [ ]:
# =============================================================================
# Validation checks
# =============================================================================

print("=" * 70)
print("VALIDATION")
print("=" * 70)

# 1. Peak ID uniqueness per species
print("\n1. Peak ID uniqueness per species:")
for species in ALL_SPECIES:
    if species not in combined_beds:
        continue
    ids = combined_beds[species]["peak_id"]
    n_unique = ids.nunique()
    n_total = len(ids)
    status = "OK" if n_unique == n_total else f"DUPLICATES: {n_total - n_unique}"
    print(f"  {species:<15s}: {n_unique:,} unique / {n_total:,} total -- {status}")

# 2. Unified peak ID consistency across species
print("\n2. Unified peak ID consistency across genomes:")
hg38_ids = set(unified_beds["Human"]["peak_id"]) if "Human" in unified_beds else set()
print(f"  Human (hg38): {len(hg38_ids):,} unified IDs")

for species in NHP_SPECIES:
    if species not in unified_beds:
        continue
    sp_ids = set(unified_beds[species]["peak_id"])
    overlap = len(hg38_ids & sp_ids)
    extra = sp_ids - hg38_ids
    missing = hg38_ids - sp_ids
    print(f"  {species:<15s}: {len(sp_ids):,} IDs, "
          f"{overlap:,} shared with hg38, "
          f"{len(missing):,} missing (liftback failed)")
    if extra:
        print(f"      \u26a0\ufe0f  {len(extra)} UNEXPECTED extra IDs!")

# 3. No overlap between unified and specific IDs
print("\n3. ID namespace separation:")
all_unified_ids = set()
all_specific_ids = set()
for species in ALL_SPECIES:
    if species not in combined_beds:
        continue
    df = combined_beds[species]
    all_unified_ids.update(df.loc[df["peak_type"] == "unified", "peak_id"])
    all_specific_ids.update(df.loc[df["peak_type"] != "unified", "peak_id"])

overlap = all_unified_ids & all_specific_ids
print(f"  Unified IDs:  {len(all_unified_ids):,}")
print(f"  Specific IDs: {len(all_specific_ids):,}")
print(f"  Overlap:      {len(overlap)} "
      f"{'-- OK (no overlap)' if not overlap else '-- WARNING: overlap!'}")

# 4. Master annotation peak type counts
print("\n4. Master annotation peak type counts:")
print(master["peak_type"].value_counts().to_string())

# 5. Detection vs orthology
print("\n5. Detection vs orthology (unified peaks):")
uni = master[master["peak_type"] == "unified"]
print(f"  {'Species':<15s} {'det':>8s} {'orth':>8s} {'orth-not-det':>14s}")
print(f"  {'-'*50}")
for sp in ALL_SPECIES:
    det_col = f"{sp}_det"
    orth_col = f"{sp}_orth"
    if det_col in uni.columns and orth_col in uni.columns:
        n_det = uni[det_col].sum()
        n_orth = uni[orth_col].sum()
        n_orth_not_det = ((uni[orth_col] == 1) & (uni[det_col] == 0)).sum()
        print(f"  {sp:<15s} {n_det:>8,} {n_orth:>8,} {n_orth_not_det:>14,}")

In [ ]:
# =============================================================================
# Summary visualizations
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Species detection distribution (unified peaks)
n_species_counts = unified_df["n_species_det"].value_counts().sort_index()
axes[0].bar(n_species_counts.index, n_species_counts.values,
            color="steelblue", edgecolor="white")
axes[0].set_xlabel("Number of species detected")
axes[0].set_ylabel("Number of peaks")
axes[0].set_title("Distribution of species detection (unified)")
for x, y in zip(n_species_counts.index, n_species_counts.values):
    axes[0].text(x, y, f"{y:,}", ha="center", va="bottom", fontsize=8)

# 2. Peak categories
categories = {"Unified": len(unified_df), "Human-specific": len(hs_df)}
for sp, n in species_specific_counts.items():
    categories[f"{sp}-specific"] = n

cats = list(categories.keys())
vals = list(categories.values())
colors_bar = ["steelblue"] + ["firebrick"] + ["seagreen"] * len(species_specific_counts)
axes[1].barh(cats, vals, color=colors_bar, edgecolor="white")
axes[1].set_xlabel("Number of peaks")
axes[1].set_title("Peak counts by category")
for y_pos, v in enumerate(vals):
    axes[1].text(v, y_pos, f"  {v:,}", ha="left", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("\n\u2705 Investigation complete.")